# Add SpatialData

- Convert the Cohort of Fobs and labels to a single `SpatialData` object. 
- Save the `SpatialData` object to `LaminDB`.


## Notebook Preferences

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Importing Libraries

In [2]:
from upath import UPath
import buckaroo  # noqa: F401
import pandas as pd
import natsort as ns
import lamindb as ln
from nbl._util import DaskLocalCluster, reset_table_index
import nbl
import spatialdata as sd

Buckaroo has been enabled as the default DataFrame viewer.  To return to default dataframe visualization use `from buckaroo import disable; disable()`
💡 connected lamindb: srivarra/nbl_assets


In [3]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", True)

In [4]:
ln.settings.transform.stem_uid = "l5ek5PQGYSZ1"
ln.settings.transform.version = "1"
run = ln.track()
run.transform

💡 notebook imports: buckaroo==0.6.12 lamindb==0.75.0 natsort==8.4.0 nbl==0.0.1 pandas==2.2.2 spatialdata==0.2.2.dev16+g6b76876 universal_pathlib==0.2.2
💡 loaded: Transform(uid='l5ek5PQGYSZ15zKv', version='1', name='Add SpatialData', key='01 - Add SpatialData', type='notebook', created_by_id=1, updated_at='2024-08-06 19:06:05 UTC')
💡 loaded: Run(uid='1anajEsM0sljIvy2h1yh', started_at='2024-08-07 05:47:29 UTC', is_consecutive=True, transform_id=3, created_by_id=1)


Transform(uid='l5ek5PQGYSZ15zKv', version='1', name='Add SpatialData', key='01 - Add SpatialData', type='notebook', created_by_id=1, updated_at='2024-08-06 19:06:05 UTC')

In [5]:
ln.setup.settings.instance

Current instance: srivarra/nbl_assets
- owner: srivarra
- name: nbl_assets
- storage root: /Users/srivarra/Davis Lab/neuroblastoma/nbl/data/db
- storage region: None
- db: postgresql://srivarra:***@***:5555/nbl_db
- schema: {'bionty'}
- git_repo: None

In [6]:
cluster = DaskLocalCluster(n_workers=10, threads_per_worker=1)
cluster(open_dashboard=True)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 10
Total threads: 10,Total memory: 64.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:56035,Workers: 10
Dashboard: http://127.0.0.1:8787/status,Total threads: 10
Started: Just now,Total memory: 64.00 GiB
Comm: tcp://127.0.0.1:56062,Total threads: 1
Dashboard: http://127.0.0.1:56086/status,Memory: 6.40 GiB
Nanny: tcp://127.0.0.1:56038,


## Convert FOVs to SpatialData

### Set up Paths

In [7]:
original_data_path = UPath("../../../data/raw/original_data/nbl_cohort")
fov_dir: UPath = original_data_path / "images"
label_dir: UPath = original_data_path / "segmentation" / "labels"

In [8]:
hu_data_path: UPath = ln.settings.storage.root / "Hu.zarr"
nbl_data_path: UPath = ln.settings.storage.root / "nbl.zarr"

### Convert Control Cohort to SpatialData - Hu Data

#### Convert Cohort

In [ ]:
nbl.io.convert_cohort(fov_dir=fov_dir, label_dir=label_dir, filter_fovs=r"Hu-*", file_path=hu_data_path)

In [ ]:
hu_sdata = sd.read_zarr(store=hu_data_path)

#### Aggregate Images by Labels and Compute Region Properties

In [ ]:
nbl.tl.aggregate_images_by_labels(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell")
nbl.tl.aggregate_images_by_labels(sdata=hu_sdata, label_type="nuclear", table_name="nuclear")

In [ ]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [ ]:
nbl.tl.regionprops(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell", properties=properties)
nbl.tl.regionprops(sdata=hu_sdata, label_type="nuclear", table_name="nuclear", properties=properties)

### Convert Sample Cohort to SpatialData - NBL Data

#### Convert Cohort

In [ ]:
nbl.io.convert_cohort(fov_dir=fov_dir, filter_fovs=r"NBL-\d+-R\d+C\d+", label_dir=label_dir, file_path=nbl_data_path)

In [9]:
nbl_sdata = sd.read_zarr(store=nbl_data_path)

#### Aggregate Images by Labels and Compute Region Properties

In [ ]:
nbl.tl.aggregate_images_by_labels(sdata=nbl_sdata, label_type="whole_cell", table_name="whole_cell")

In [ ]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [ ]:
nbl.tl.regionprops(sdata=nbl_sdata, label_type="whole_cell", table_name="whole_cell", properties=properties)

## Add Pixie Clusters to NBL SpatialData

In [10]:
pixie_clusters_path = (
    original_data_path / "segmentation" / "cell_table" / "cell_table_size_normalized_cell_labels_noCD117.csv"
)

In [11]:
pixie_clusters_df = pd.read_csv(pixie_clusters_path).astype({"label": int, "cell_meta_cluster": "category"})

In [12]:
all_fovs = ns.natsorted(nbl_sdata.coordinate_systems)

In [14]:
def get_pixie_clusters(df, fovs: str, suffix="whole_cell") -> pd.DataFrame:
    """Gets pixie clusters from the two clustering csv files.

    Parameters
    ----------
    df : pd.DataFrame
        The Pixie cluster DataFrame
    fov : str
        The FOV to subset on

    Returns
    -------
    pd.DataFrame
        A dataframe containing the two clusters and a column for the segmentation label.
    """
    out_df = []
    for fov in fovs:
        fov_rncm = fov.split("-")[-1].split("_")[0]
        no_cd117_pixie: pd.DataFrame = df[df["fov"].str.split("_").map(lambda x: x[-1]) == fov_rncm]
        result_df = (
            no_cd117_pixie.assign(region=f"{fov}_{suffix}", fov=fov)
            .rename(columns={"label": "instance_id", "cell_meta_cluster": "pixie_cluster"})
            .astype(dtype={"instance_id": int, "region": "category", "pixie_cluster": "category"})
            .filter(items=["instance_id", "region", "fov", "pixie_cluster"])
        )
        out_df.append(result_df)

    return pd.concat(out_df)

In [15]:
pixie_df = pixie_clusters_df.pipe(func=get_pixie_clusters, fovs=all_fovs, suffix="whole_cell")

In [17]:
nbl_sdata.tables["whole_cell"].obs = (
    nbl_sdata.tables["whole_cell"]
    .obs.merge(
        right=pixie_df,
        right_on=["instance_id", "region"],
        left_on=["instance_id", "region"],
    )
    .pipe(reset_table_index)
)

In [18]:
nbl_sdata.tables["whole_cell"].obs

BuckarooWidget(buckaroo_options={'sampled': ['random'], 'auto_clean': ['aggressive', 'conservative'], 'post_pr…

## Add Clinical Information

### Load Clinical Data from LaminDB

In [19]:
clinical_data: pd.DataFrame = ln.Artifact.filter(key__contains="clinical_data").one().load()

In [20]:
cols_to_drop = ["Clinical presentation", "treatment btw biopsies"]

In [21]:
filtered_clincial_data = clinical_data.drop(columns=cols_to_drop)

In [22]:
nbl_sdata.tables["whole_cell"].obs = (
    nbl_sdata.tables["whole_cell"].obs.merge(right=filtered_clincial_data, on="fov").pipe(reset_table_index)
)
nbl_sdata.tables["whole_cell"].strings_to_categoricals()

### `Arcsinh` Transform the NBL Whole Cell Table

In [24]:
nbl.pp.arcsinh_transform(
    sdata=nbl_sdata, table_names="whole_cell", shift_factor=5, scale_factor=150, replace_X=True, write=True
)

/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/_core/_elements.py:116: UserWarning: Key `whole_cell__temp` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


In [25]:
hu_artifact = ln.Artifact(data=hu_data_path, type="dataset", key="Hu.zarr", description="Control Tissue")

hu_artifact.save(upload=True)

Artifact(uid='BN6lN7oaDL4IyTMflmSz', description='Control Tissue', key='Hu.zarr', suffix='.zarr', type='dataset', size=434280085, hash='ILejb2CtOyUNq0ybtDHJXQ', _hash_type='md5-d', n_objects=466, visibility=1, _key_is_virtual=False, created_by_id=1, storage_id=1, transform_id=3, run_id=3, updated_at='2024-08-07 06:15:03 UTC')

In [26]:
nbl_artifact = ln.Artifact(data=nbl_data_path, type="dataset", key="nbl.zarr", description="NBL Tissue Samples")

nbl_artifact.save(upload=True)

Artifact(uid='SFEXPE9JArITna7dLRJc', description='NBL Tissue Samples', key='nbl.zarr', suffix='.zarr', type='dataset', size=12716207886, hash='0k_BwWK-E8hZPdQdKue9_Q', _hash_type='md5-d', n_objects=6638, visibility=1, _key_is_virtual=False, created_by_id=1, storage_id=1, transform_id=3, run_id=3, updated_at='2024-08-07 06:15:11 UTC')

In [28]:
ln.finish()

❗ cells [(8, None), (None, None), (None, None), (None, None), (None, None), (None, None), (None, 9), (9, None), (None, None), (None, None), (None, 10), (12, 14), (15, 17), (22, 24)] were not run consecutively
